In [49]:
import requests
import pandas as pd

pd.set_option("display.max_columns", None)

In [50]:
API_KEY = "jobboerse-jobsuche"
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/app/jobs"

headers = {
    "X-API-Key": API_KEY
}

### Erste Datensammlung

Es werden mehrere Suchbegriffe verwendet, um verschiedene Berufsfelder abzudecken. Pro Suchbegriff werden mehrere Seiten geladen. Ziel ist noch nicht ein perfekter Enddatensatz, sondern ein erster Rohdatensatz für die Datenexploration.

In [51]:
search_terms = ["data", "marketing", "logistik", "verkauf", "software", "pflege"]

all_jobs = []

for term in search_terms:
    print("Lade Daten für:", term)

    for page in range(1, 4):
        params = {
            "was": term,
            "page": page,
            "size": 100
        }

        response = requests.get(BASE_URL, headers=headers, params=params)

        if response.status_code != 200:
            print("Fehler bei", term, "Seite", page, "Status:", response.status_code)
            continue

        data = response.json()
        jobs = data.get("stellenangebote", [])

        print("  Seite", page, "-", len(jobs), "Anzeigen")

        for job in jobs:
            all_jobs.append(job)

print("\nGesamtzahl geladener Rohdatensätze:", len(all_jobs))

Lade Daten für: data
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: marketing
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: logistik
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: verkauf
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: software
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: pflege
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen

Gesamtzahl geladener Rohdatensätze: 1800


In [52]:
if len(all_jobs) > 0:
    print("Beispielhafte Keys eines Job-Objekts:\n")
    print(all_jobs[0].keys())

Beispielhafte Keys eines Job-Objekts:

dict_keys(['beruf', 'titel', 'refnr', 'arbeitsort', 'arbeitgeber', 'aktuelleVeroeffentlichungsdatum', 'modifikationsTimestamp', 'eintrittsdatum', 'kundennummerHash'])


In [53]:
def get_nested_value(d, keys, default=None):
    value = d
    for key in keys:
        if isinstance(value, dict) and key in value:
            value = value[key]
        else:
            return default
    return value


rows = []

for job in all_jobs:
    row = {
        "id": job.get("hashId"),
        "titel": job.get("titel"),
        "beruf": job.get("beruf"),
        "arbeitgeber": job.get("arbeitgeber"),
        "arbeitszeitmodell": job.get("arbeitszeitmodell"),
        "eintrittsdatum": job.get("eintrittsdatum"),
        "aktualisiert_bis": job.get("aktuellBis"),
        "ort": get_nested_value(job, ["arbeitsort", "ort"]),
        "region": get_nested_value(job, ["arbeitsort", "region"]),
        "plz": get_nested_value(job, ["arbeitsort", "plz"]),
        "beschreibung": job.get("stellenbeschreibung")
    }
    rows.append(row)

df = pd.DataFrame(rows)

print("Form des DataFrames:", df.shape)
df.head()

Form des DataFrames: (1800, 11)


,id,titel,beruf,arbeitgeber,arbeitszeitmodell,eintrittsdatum,aktualisiert_bis,ort,region,plz,beschreibung
0,None,Data Engineer (m/w/d),Data Engineer,FERCHAU GmbH Niederlassung Bielefeld,None,2026-04-01,None,Bielefeld,Nordrhein-Westfalen,33602,None
1,None,Data Engineer (m/w/d),Data Engineer,FERCHAU GmbH Niederlassung Rosenheim,None,2026-03-31,None,"Engelsberg, Oberbayern",Bayern,84549,None
2,None,Data Engineer (m/w/d),Data Engineer,BTB Blockheizkraftwerks- Träger- und Betreiber...,None,2026-04-01,None,Berlin,Berlin,10585,None
3,None,Data Engineer (m/w/d),Data Engineer,Riverty Group GmbH,None,2026-04-01,None,Berlin,Berlin,10623,None
4,None,Data Engineer,Data Engineer,PwC Transaction Services Wirtschaftsprüfung GmbH,None,2026-04-01,None,"Wien,Donaustadt",Wien,1220,None
